# Directed GNN + Min-Cost Flow: Yön Bilgisi Neden Önemli?

Bu notebook, yöneylem araştırmasında **yönlü graph yapısının** neden ayrı ele alınması gerektiğini gösterir. Aynı directed min-cost-flow örneklerinde iki yaklaşım karşılaştırılır:

1. **Undirected/symmetrized GNN:** `u -> v` ve `v -> u` ayrımını mesajlaşmada kaybeder.
2. **Directed GNN:** incoming ve outgoing mesajları ayrı toplar.

Akış: `directed network -> full min-cost-flow LP -> optimal arc labels -> GNN arc scores -> candidate arc pruning -> feasibility fallback -> küçültülmüş LP`.

> GNN solver'ın yerine geçmez. GNN yalnızca aday arc'ları sıralar; feasibility ve nihai objective klasik optimizasyon modeli tarafından doğrulanır.


In [ ]:
import random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from scipy.optimize import linprog
from torch_geometric.data import Data

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## 1. Directed min-cost-flow modeli

Katmanlı ağ: `source -> L1 -> L2 -> sink`. Her arc'ın maliyeti ve kapasitesi vardır. Amaç `min sum(c_ij x_ij)`; her düğümde flow-balance, her arc'ta `0 <= x_ij <= u_ij` uygulanır.


In [ ]:
def generate_instance(n_l1=5, n_l2=5, density=0.55, total_flow=10.0, seed=0):
    rng = np.random.default_rng(seed)
    source = 0; l1 = list(range(1, 1+n_l1)); l2 = list(range(1+n_l1, 1+n_l1+n_l2))
    sink = 1+n_l1+n_l2; n_nodes = sink+1; arcs = []
    def add(u, v, cl, ch, ul, uh):
        arcs.append((u, v, float(rng.uniform(cl, ch)), float(rng.uniform(ul, uh))))
    for u in l1: add(source, u, 1, 4, total_flow*.35, total_flow*.75)
    for v in l2: add(v, sink, 1, 4, total_flow*.35, total_flow*.75)
    for u in l1:
        chosen = []
        for v in l2:
            if rng.random() < density:
                add(u, v, 1, 8, total_flow*.2, total_flow*.7); chosen.append(v)
        if not chosen: add(u, int(rng.choice(l2)), 1, 8, total_flow*.2, total_flow*.7)
    incoming = {v: 0 for v in l2}
    for u, v, _, _ in arcs:
        if u in l1 and v in incoming: incoming[v] += 1
    for v, deg in incoming.items():
        if deg == 0: add(int(rng.choice(l1)), v, 1, 8, total_flow*.2, total_flow*.7)
    for k in range(min(n_l1, n_l2)):
        u, v = l1[k], l2[k]
        if not any(a == u and b == v for a, b, _, _ in arcs): add(u, v, 1, 3, total_flow, total_flow*1.2)
    balance = np.zeros(n_nodes); balance[source] = total_flow; balance[sink] = -total_flow
    stage = np.zeros(n_nodes, dtype=int); stage[l1] = 1; stage[l2] = 2; stage[sink] = 3
    return {'n_nodes': n_nodes, 'arcs': arcs, 'balance': balance, 'stage': stage}

def solve_flow(inst, keep_mask=None):
    arcs = inst['arcs']; n = inst['n_nodes']; m = len(arcs)
    if keep_mask is None: keep_mask = np.ones(m, dtype=bool)
    idx = np.where(np.asarray(keep_mask, dtype=bool))[0]
    if len(idx) == 0: return None
    c = np.array([arcs[k][2] for k in idx])
    Aeq = np.zeros((n, len(idx))); bounds = []
    for col, k in enumerate(idx):
        u, v, _, cap = arcs[k]; Aeq[u, col] += 1; Aeq[v, col] -= 1; bounds.append((0, cap))
    t0 = time.perf_counter()
    res = linprog(c, A_eq=Aeq, b_eq=inst['balance'], bounds=bounds, method='highs')
    elapsed = time.perf_counter() - t0
    if not res.success: return None
    flow = np.zeros(m); flow[idx] = res.x
    return {'objective': float(res.fun), 'flow': flow, 'solve_time': elapsed, 'kept_arcs': len(idx)}

example = generate_instance(seed=SEED)
full = solve_flow(example)
full['objective'], np.count_nonzero(full['flow'] > 1e-8)


## 2. Graph feature'ları

Node feature'ları: arz/talep, stage one-hot, in-degree ve out-degree. Edge feature'ları: maliyet ve kapasite. Etiket `y_e=1`, eğer full optimum arc'ı pozitif akışla kullanıyorsa.


In [ ]:
def scale(x):
    x = np.asarray(x, dtype=float); m = np.max(np.abs(x)); return x/m if m > 0 else x

def to_data(inst):
    sol = solve_flow(inst)
    arcs = inst['arcs']; n = inst['n_nodes']
    indeg = np.zeros(n); outdeg = np.zeros(n)
    for u, v, _, _ in arcs: outdeg[u] += 1; indeg[v] += 1
    stage_oh = np.eye(4)[inst['stage']]
    x = np.column_stack([scale(inst['balance']), stage_oh, indeg/max(1, indeg.max()), outdeg/max(1, outdeg.max())])
    src = np.array([a[0] for a in arcs]); dst = np.array([a[1] for a in arcs])
    costs = np.array([a[2] for a in arcs]); caps = np.array([a[3] for a in arcs])
    edge_attr = np.column_stack([scale(costs), scale(caps)])
    y = (sol['flow'] > 1e-8).astype(float)
    return Data(x=torch.tensor(x, dtype=torch.float32), edge_index=torch.tensor(np.vstack([src, dst]), dtype=torch.long), edge_attr=torch.tensor(edge_attr, dtype=torch.float32), y=torch.tensor(y, dtype=torch.float32))

to_data(example)


## 3. Undirected ve Directed message passing

Undirected model edge'leri simetrize eder. Directed model ise incoming ve outgoing mesajları ayrı kanallarda toplar; böylece yön semantiğini korur.


In [ ]:
def mean_agg(msg, index, n):
    out = msg.new_zeros((n, msg.size(-1))); out.index_add_(0, index, msg)
    cnt = msg.new_zeros((n, 1)); cnt.index_add_(0, index, msg.new_ones((msg.size(0), 1)))
    return out / cnt.clamp_min(1)

class UndirectedBlock(nn.Module):
    def __init__(self, h, e):
        super().__init__(); self.msg = nn.Sequential(nn.Linear(h+e, h), nn.ReLU(), nn.Linear(h, h)); self.upd = nn.Linear(2*h, h); self.norm = nn.LayerNorm(h)
    def forward(self, h, edge_index, edge_attr):
        s, d = edge_index; ss = torch.cat([s, d]); dd = torch.cat([d, s]); ee = torch.cat([edge_attr, edge_attr])
        agg = mean_agg(self.msg(torch.cat([h[ss], ee], -1)), dd, h.size(0))
        return self.norm(h + F.relu(self.upd(torch.cat([h, agg], -1))))

class DirectedBlock(nn.Module):
    def __init__(self, h, e):
        super().__init__(); self.mi = nn.Sequential(nn.Linear(h+e, h), nn.ReLU(), nn.Linear(h, h)); self.mo = nn.Sequential(nn.Linear(h+e, h), nn.ReLU(), nn.Linear(h, h)); self.upd = nn.Linear(3*h, h); self.norm = nn.LayerNorm(h)
    def forward(self, h, edge_index, edge_attr):
        s, d = edge_index
        ain = mean_agg(self.mi(torch.cat([h[s], edge_attr], -1)), d, h.size(0))
        aout = mean_agg(self.mo(torch.cat([h[d], edge_attr], -1)), s, h.size(0))
        return self.norm(h + F.relu(self.upd(torch.cat([h, ain, aout], -1))))

class ArcScorer(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=64, directed=True):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(node_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        block = DirectedBlock if directed else UndirectedBlock; self.blocks = nn.ModuleList([block(hidden, edge_dim), block(hidden, edge_dim)])
        self.head = nn.Sequential(nn.Linear(2*hidden+edge_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, data):
        h = self.enc(data.x)
        for b in self.blocks: h = b(h, data.edge_index, data.edge_attr)
        s, d = data.edge_index
        return self.head(torch.cat([h[s], h[d], data.edge_attr], -1)).squeeze(-1)


## 4. Eğitim ve karşılaştırma

Arc kullanımı dengesiz olabildiği için positive-class ağırlıklı BCE kullanılır. Gerçek değerlendirmede accuracy'den çok optimal-arc recall, feasibility ve objective gap önemlidir.


In [ ]:
instances = [generate_instance(density=np.random.default_rng(1000+k).uniform(.4, .75), seed=1000+k) for k in range(120)]
dataset = [to_data(inst) for inst in instances]
train, val, test = dataset[:80], dataset[80:100], dataset[100:]
train_inst, test_inst = instances[:80], instances[100:]

def train_model(model, epochs=60):
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-5)
    ys = torch.cat([d.y for d in train]); pos = ys.sum().item(); pw = torch.tensor((len(ys)-pos)/max(pos,1), device=device)
    best, best_loss = None, float('inf')
    for ep in range(epochs):
        model.train()
        for i in np.random.permutation(len(train)):
            d = train[i].to(device); opt.zero_grad(); loss = F.binary_cross_entropy_with_logits(model(d), d.y, pos_weight=pw); loss.backward(); opt.step()
        model.eval(); losses = []
        with torch.no_grad():
            for d0 in val:
                d = d0.to(device); losses.append(F.binary_cross_entropy_with_logits(model(d), d.y, pos_weight=pw).item())
        vl = np.mean(losses)
        if vl < best_loss: best_loss = vl; best = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        if ep == 0 or (ep+1) % 10 == 0: print(ep+1, vl)
    model.load_state_dict(best); return model

nd, ed = train[0].x.size(-1), train[0].edge_attr.size(-1)
undirected = train_model(ArcScorer(nd, ed, directed=False))
directed = train_model(ArcScorer(nd, ed, directed=True))


In [ ]:
def metrics(model):
    tp=fp=tn=fn=0; model.eval()
    with torch.no_grad():
        for d0 in test:
            d=d0.to(device); p=(torch.sigmoid(model(d))>=.5).cpu().numpy(); y=d0.y.numpy().astype(bool)
            tp += int((p & y).sum()); fp += int((p & ~y).sum()); tn += int((~p & ~y).sum()); fn += int((~p & y).sum())
    return {'accuracy':(tp+tn)/max(tp+tn+fp+fn,1), 'precision':tp/max(tp+fp,1), 'recall':tp/max(tp+fn,1)}

print('Undirected:', metrics(undirected))
print('Directed  :', metrics(directed))


## 5. Arc pruning + feasibility fallback

Arc'lar model skoruna göre sıralanır. İlk etapta yalnızca üst dilim tutulur; model infeasible olursa tutulan arc oranı kademeli artırılır. Böylece GNN feasibility doğrulamasının yerine geçmez.


In [ ]:
def scores(model, data):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(data.to(device))).cpu().numpy()

def prune_and_solve(inst, s, initial=.45, step=.10):
    m=len(inst['arcs']); order=np.argsort(-s); ratio=initial
    while ratio <= 1.00001:
        k=max(1, int(np.ceil(m*min(ratio,1)))); keep=np.zeros(m,dtype=bool); keep[order[:k]]=True
        sol=solve_flow(inst, keep)
        if sol is not None: sol['keep_ratio']=k/m; return sol
        ratio += step
    sol=solve_flow(inst); sol['keep_ratio']=1.0; return sol

def cost_only(inst):
    c=np.array([a[2] for a in inst['arcs']]); u=np.array([a[3] for a in inst['arcs']])
    return 1-c/max(c.max(),1e-9)+.25*u/max(u.max(),1e-9)

def evaluate(model=None, name='cost_only'):
    rows=[]
    for d, inst in zip(test, test_inst):
        full=solve_flow(inst); s=cost_only(inst) if model is None else scores(model,d); pr=prune_and_solve(inst,s)
        gap=(pr['objective']-full['objective'])/max(abs(full['objective']),1e-9)
        rows.append((name, gap, pr['keep_ratio'], full['solve_time'], pr['solve_time']))
    return rows

rows = evaluate(undirected,'undirected_gnn') + evaluate(directed,'directed_gnn') + evaluate(None,'cost_only')
for method in ['undirected_gnn','directed_gnn','cost_only']:
    rr=[r for r in rows if r[0]==method]
    print(method, {'mean_gap_%':100*np.mean([r[1] for r in rr]), 'mean_keep_%':100*np.mean([r[2] for r in rr]), 'mean_pruned_ms':1000*np.mean([r[4] for r in rr])})


## 6. Yorum

Directed GNN'nin anlamlı olup olmadığını şu sorularla değerlendirin: optimal arc recall daha yüksek mi, aynı feasibility düzeyinde daha az arc tutuluyor mu, objective gap düşüyor mu ve cost-only heuristic'e karşı ek değer sağlıyor mu?

Yön bilgisi özellikle minimum-cost flow, transportation/transshipment, supply-chain network design, precedence-constrained scheduling, project networks, tek yönlü routing ve telecommunication/energy flow problemlerinde problem semantiğinin parçasıdır.

> Ana fikir: Directed GNN solver'ın yerine geçmez; yönlü karar uzayında daha iyi candidate filtering yapmaya çalışır.
